In [1]:
from pathlib import Path
from collections import Counter

import pandas as pd
import numpy as np

from bs4 import BeautifulSoup

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 300)

PROJECT_DIR = Path("..").resolve()

REPORTS_DIR = (
    PROJECT_DIR
    / "reports"
    / "tables"
)

INTERIM_DIR = (
    PROJECT_DIR
    / "data"
    / "public_v2"
    / "interim"
)

confirmed_registry = pd.read_csv(
    REPORTS_DIR
    / "catching_confirmed_repository_registry.csv"
)

html_inventory = pd.read_csv(
    REPORTS_DIR
    / "catching_confirmed_html_inventory.csv"
)

direct_registry = pd.read_csv(
    INTERIM_DIR
    / "catching_direct_validation_registry.csv"
)

print(
    confirmed_registry.shape,
    html_inventory.shape,
    direct_registry.shape
)

(2, 16) (15, 5) (2, 16)


In [2]:
static_feature_semantics = pd.DataFrame([
    {
        "static_variable":
            "static_dom_max_depth",

        "related_construct":
            "STRUCTURAL_COMPLEXITY",

        "possible_relation":
            "H6",

        "canonical_equivalent":
            False,

        "must_not_call":
            "navigation_depth",

        "reason":
            "Profundidad HTML no equivale a pasos de navegación."
    },

    {
        "static_variable":
            "static_control_label_proxy_ratio",

        "related_construct":
            "CONTROL_RECOGNIZABILITY",

        "possible_relation":
            "H6",

        "canonical_equivalent":
            False,

        "must_not_call":
            "accessible_name_ratio / visible_labels_ratio",

        "reason":
            "HTML estático no permite reproducir completamente Accessible Name ni visibilidad renderizada."
    },

    {
        "static_variable":
            "static_native_validation_ratio",

        "related_construct":
            "ERROR_PREVENTION_CAPABILITY",

        "possible_relation":
            "H5",

        "canonical_equivalent":
            False,

        "must_not_call":
            "client_validation_present",

        "reason":
            "required/min/max/pattern/type solo evidencian validación nativa declarada."
    },

    {
        "static_variable":
            "static_class_reuse_ratio",

        "related_construct":
            "STRUCTURAL_REUSE",

        "possible_relation":
            "H4",

        "canonical_equivalent":
            False,

        "must_not_call":
            "component_style_consistency_ratio",

        "reason":
            "Reutilización de clases no demuestra consistencia visual."
    }
])

static_feature_semantics

,static_variable,related_construct,possible_relation,canonical_equivalent,must_not_call,reason
0,static_dom_max_depth,STRUCTURAL_COMPLEXITY,H6,False,navigation_depth,Profundidad HTML no equivale a pasos de navegación.
1,static_control_label_proxy_ratio,CONTROL_RECOGNIZABILITY,H6,False,accessible_name_ratio / visible_labels_ratio,HTML estático no permite reproducir completamente Accessible Name ni visibilidad renderizada.
2,static_native_validation_ratio,ERROR_PREVENTION_CAPABILITY,H5,False,client_validation_present,required/min/max/pattern/type solo evidencian validación nativa declarada.
3,static_class_reuse_ratio,STRUCTURAL_REUSE,H4,False,component_style_consistency_ratio,Reutilización de clases no demuestra consistencia visual.


In [3]:
def element_depth(
    element,
    current_depth=0
):

    max_depth = current_depth

    for child in element.children:

        if getattr(
            child,
            "name",
            None
        ) is None:
            continue

        child_depth = element_depth(
            child,
            current_depth + 1
        )

        max_depth = max(
            max_depth,
            child_depth
        )

    return max_depth

In [4]:
INTERACTIVE_TAGS = {
    "a",
    "button",
    "input",
    "select",
    "textarea"
}

In [5]:
def get_interactive_elements(
    soup
):

    controls = []

    for element in soup.find_all(
        list(
            INTERACTIVE_TAGS
        )
    ):

        if (
            element.name == "input"
            and
            str(
                element.get(
                    "type",
                    ""
                )
            ).lower()
            ==
            "hidden"
        ):
            continue

        controls.append(
            element
        )

    return controls

In [6]:
def control_has_static_label(
    element,
    soup
):

    # Texto del propio control
    if element.get_text(
        " ",
        strip=True
    ):
        return True

    # aria-label
    if str(
        element.get(
            "aria-label",
            ""
        )
    ).strip():
        return True

    # title
    if str(
        element.get(
            "title",
            ""
        )
    ).strip():
        return True

    # value en botones/input submit
    if str(
        element.get(
            "value",
            ""
        )
    ).strip():
        return True

    # alt
    if str(
        element.get(
            "alt",
            ""
        )
    ).strip():
        return True

    # <label for=id>
    element_id = element.get(
        "id"
    )

    if element_id:

        label = soup.find(
            "label",
            attrs={
                "for":
                    element_id
            }
        )

        if (
            label is not None
            and
            label.get_text(
                " ",
                strip=True
            )
        ):
            return True

    # Control contenido dentro de <label>
    parent_label = element.find_parent(
        "label"
    )

    if (
        parent_label is not None
        and
        parent_label.get_text(
            " ",
            strip=True
        )
    ):
        return True

    return False

In [7]:
VALIDATION_ATTRIBUTES = {
    "required",
    "pattern",
    "min",
    "max",
    "minlength",
    "maxlength"
}

VALIDATING_INPUT_TYPES = {
    "email",
    "url",
    "number",
    "date",
    "datetime-local",
    "month",
    "week",
    "time"
}

In [8]:
def has_native_validation(
    element
):

    if any(
        attr in element.attrs
        for attr in VALIDATION_ATTRIBUTES
    ):
        return True

    if (
        element.name == "input"
        and
        str(
            element.get(
                "type",
                ""
            )
        ).lower()
        in VALIDATING_INPUT_TYPES
    ):
        return True

    return False

In [9]:
def class_reuse_metrics(
    soup
):

    class_tokens = []

    for element in soup.find_all(
        True
    ):

        classes = element.get(
            "class",
            []
        )

        if not isinstance(
            classes,
            list
        ):
            continue

        class_tokens.extend(
            str(x)
            for x in classes
            if str(x).strip()
        )

    if not class_tokens:

        return {
            "class_token_count": 0,
            "unique_class_count": 0,
            "static_class_reuse_ratio":
                np.nan
        }

    counts = Counter(
        class_tokens
    )

    reused_occurrences = sum(
        count
        for count
        in counts.values()
        if count >= 2
    )

    return {
        "class_token_count":
            len(class_tokens),

        "unique_class_count":
            len(counts),

        "static_class_reuse_ratio":
            reused_occurrences
            /
            len(class_tokens)
    }

In [10]:
def extract_static_page_features(
    html_path
):

    text = html_path.read_text(
        encoding="utf-8",
        errors="ignore"
    )

    soup = BeautifulSoup(
        text,
        "html.parser"
    )

    all_elements = soup.find_all(
        True
    )

    controls = get_interactive_elements(
        soup
    )

    labeled_controls = [
        control
        for control in controls
        if control_has_static_label(
            control,
            soup
        )
    ]

    form_controls = [
        control
        for control in controls
        if control.name
        in {
            "input",
            "select",
            "textarea"
        }
    ]

    validated_controls = [
        control
        for control in form_controls
        if has_native_validation(
            control
        )
    ]

    class_metrics = (
        class_reuse_metrics(
            soup
        )
    )

    return {
        "static_element_count":
            len(
                all_elements
            ),

        "static_dom_max_depth":
            (
                element_depth(
                    soup.html,
                    0
                )
                if soup.html
                else np.nan
            ),

        "static_interactive_control_count":
            len(
                controls
            ),

        "static_labeled_control_count":
            len(
                labeled_controls
            ),

        "static_control_label_proxy_ratio":
            (
                len(
                    labeled_controls
                )
                /
                len(
                    controls
                )
                if controls
                else np.nan
            ),

        "static_form_control_count":
            len(
                form_controls
            ),

        "static_native_validated_control_count":
            len(
                validated_controls
            ),

        "static_native_validation_ratio":
            (
                len(
                    validated_controls
                )
                /
                len(
                    form_controls
                )
                if form_controls
                else np.nan
            ),

        **class_metrics
    }

In [11]:
page_feature_records = []

path_lookup = (
    confirmed_registry
    .set_index("repo_full_name")[
        "local_path"
    ]
    .to_dict()
)

for _, row in html_inventory.iterrows():

    repo = row[
        "repo_full_name"
    ]

    repo_dir = Path(
        path_lookup[
            repo
        ]
    )

    html_path = (
        repo_dir
        /
        row[
            "relative_path"
        ]
    )

    features = (
        extract_static_page_features(
            html_path
        )
    )

    page_feature_records.append({
        "site":
            row[
                "site"
            ],

        "repo_full_name":
            repo,

        "relative_path":
            row[
                "relative_path"
            ],

        **features
    })

catching_page_features = pd.DataFrame(
    page_feature_records
)

In [12]:
print(
    catching_page_features.shape
)

print(
    catching_page_features[
        "site"
    ].value_counts()
)

display(
    catching_page_features
)

(15, 14)
site
startbootstrap-sb-admin-2         14
startbootstrap-3-col-portfolio     1
Name: count, dtype: int64


,site,repo_full_name,relative_path,static_element_count,static_dom_max_depth,static_interactive_control_count,static_labeled_control_count,static_control_label_proxy_ratio,static_form_control_count,static_native_validated_control_count,static_native_validation_ratio,class_token_count,unique_class_count,static_class_reuse_ratio
0,startbootstrap-3-col-portfolio,StartBootstrap/startbootstrap-3-col-portfolio,index.html,98,8,23,17,0.739130,0,0,NaN,100,36,0.800000
1,startbootstrap-sb-admin-2,StartBootstrap/startbootstrap-sb-admin-2,buttons.html,302,13,71,50,0.704225,2,0,0.000000,575,170,0.867826
2,startbootstrap-sb-admin-2,StartBootstrap/startbootstrap-sb-admin-2,utilities-border.html,227,13,46,40,0.869565,2,0,0.000000,435,157,0.790805
3,startbootstrap-sb-admin-2,StartBootstrap/startbootstrap-sb-admin-2,index.html,362,15,57,49,0.859649,2,0,0.000000,737,206,0.857531
4,startbootstrap-sb-admin-2,StartBootstrap/startbootstrap-sb-admin-2,forgot-password.html,37,12,4,3,0.750000,1,1,1.000000,38,33,0.236842
5,startbootstrap-sb-admin-2,StartBootstrap/startbootstrap-sb-admin-2,utilities-color.html,244,13,46,40,0.869565,2,0,0.000000,491,173,0.802444
6,startbootstrap-sb-admin-2,StartBootstrap/startbootstrap-sb-admin-2,register.html,49,11,10,5,0.500000,5,1,0.200000,66,38,0.636364
7,startbootstrap-sb-admin-2,StartBootstrap/startbootstrap-sb-admin-2,404.html,198,13,46,40,0.869565,2,0,0.000000,366,141,0.781421
8,startbootstrap-sb-admin-2,StartBootstrap/startbootstrap-sb-admin-2,login.html,47,13,8,6,0.750000,3,1,0.333333,60,42,0.500000
9,startbootstrap-sb-admin-2,StartBootstrap/startbootstrap-sb-admin-2,blank.html,194,13,45,39,0.866667,2,0,0.000000,361,137,0.792244


In [13]:
assert (
    catching_page_features.shape[0]
    ==
    15
)

print(
    "Las Y NO se unirán a nivel página ✅"
)

Las Y NO se unirán a nivel página ✅


In [14]:
def aggregate_site_features(
    group
):

    interactive_total = (
        group[
            "static_interactive_control_count"
        ].sum()
    )

    labeled_total = (
        group[
            "static_labeled_control_count"
        ].sum()
    )

    form_total = (
        group[
            "static_form_control_count"
        ].sum()
    )

    validated_total = (
        group[
            "static_native_validated_control_count"
        ].sum()
    )

    return pd.Series({
        "static_page_count":
            len(group),

        "static_mean_element_count":
            group[
                "static_element_count"
            ].mean(),

        "static_max_element_count":
            group[
                "static_element_count"
            ].max(),

        "static_mean_dom_depth":
            group[
                "static_dom_max_depth"
            ].mean(),

        "static_max_dom_depth":
            group[
                "static_dom_max_depth"
            ].max(),

        "static_total_interactive_controls":
            interactive_total,

        "static_control_label_proxy_ratio":
            (
                labeled_total
                /
                interactive_total
                if interactive_total > 0
                else np.nan
            ),

        "static_total_form_controls":
            form_total,

        "static_native_validation_ratio":
            (
                validated_total
                /
                form_total
                if form_total > 0
                else np.nan
            ),

        "static_mean_class_reuse_ratio":
            group[
                "static_class_reuse_ratio"
            ].mean()
    })

In [15]:
catching_site_features = (
    catching_page_features
    .groupby(
        [
            "site",
            "repo_full_name"
        ]
    )
    .apply(
        aggregate_site_features,
        include_groups=False
    )
    .reset_index()
)

In [16]:
print(
    catching_site_features.shape
)

display(
    catching_site_features
)

assert (
    catching_site_features[
        "site"
    ].nunique()
    ==
    2
)

(2, 12)


,site,repo_full_name,static_page_count,static_mean_element_count,static_max_element_count,static_mean_dom_depth,static_max_dom_depth,static_total_interactive_controls,static_control_label_proxy_ratio,static_total_form_controls,static_native_validation_ratio,static_mean_class_reuse_ratio
0,startbootstrap-3-col-portfolio,StartBootstrap/startbootstrap-3-col-portfolio,1.0,98.000000,98.0,8.000000,8.0,23.0,0.739130,0.0,NaN,0.800000
1,startbootstrap-sb-admin-2,StartBootstrap/startbootstrap-sb-admin-2,14.0,232.357143,623.0,13.142857,15.0,585.0,0.842735,31.0,0.096774,0.737178


In [17]:
catching_static_validation = (
    catching_site_features
    .merge(
        direct_registry[
            [
                "site",
                "commit_sha",
                "label_source",
                "label_level",
                "H1_violation",
                "H4_violation",
                "H5_violation",
                "H6_violation"
            ]
        ],
        on="site",
        how="inner",
        validate="one_to_one"
    )
)

In [18]:
print(
    catching_static_validation.shape
)

display(
    catching_static_validation
)

(2, 19)


,site,repo_full_name,static_page_count,static_mean_element_count,static_max_element_count,static_mean_dom_depth,static_max_dom_depth,static_total_interactive_controls,static_control_label_proxy_ratio,static_total_form_controls,static_native_validation_ratio,static_mean_class_reuse_ratio,commit_sha,label_source,label_level,H1_violation,H4_violation,H5_violation,H6_violation
0,startbootstrap-3-col-portfolio,StartBootstrap/startbootstrap-3-col-portfolio,1.0,98.000000,98.0,8.000000,8.0,23.0,0.739130,0.0,NaN,0.800000,f544d6140b13133be8953037fe4fb09a54172ea9,LLM_CONSENSUS,LLM_DIRECT,1,1.0,1,1
1,startbootstrap-sb-admin-2,StartBootstrap/startbootstrap-sb-admin-2,14.0,232.357143,623.0,13.142857,15.0,585.0,0.842735,31.0,0.096774,0.737178,f0309881ef82794a1bd6257cd321801bc38a0f3d,LLM_CONSENSUS,LLM_DIRECT,1,1.0,1,1


In [19]:
catching_static_validation[
    "source_dataset"
] = "Catching_UX_Flaws"

catching_static_validation[
    "feature_modality"
] = "STATIC_HTML_SOURCE"

catching_static_validation[
    "primary_training"
] = False

catching_static_validation[
    "intended_use"
] = "PIPELINE_VALIDATION"

In [20]:
catching_page_features.to_csv(
    INTERIM_DIR
    / "catching_static_page_features.csv",
    index=False
)

catching_site_features.to_csv(
    INTERIM_DIR
    / "catching_static_site_features.csv",
    index=False
)

catching_static_validation.to_csv(
    INTERIM_DIR
    / "catching_static_validation_dataset.csv",
    index=False
)

static_feature_semantics.to_csv(
    REPORTS_DIR
    / "catching_static_feature_semantics.csv",
    index=False
)

In [21]:
for filename in [
    "catching_static_page_features.csv",
    "catching_static_site_features.csv",
    "catching_static_validation_dataset.csv"
]:

    path = (
        INTERIM_DIR
        / filename
    )

    print(
        filename,
        "✅" if path.exists()
        else "❌"
    )

print(
    "catching_static_feature_semantics.csv",
    "✅"
    if (
        REPORTS_DIR
        / "catching_static_feature_semantics.csv"
    ).exists()
    else "❌"
)

catching_static_page_features.csv ✅
catching_static_site_features.csv ✅
catching_static_validation_dataset.csv ✅
catching_static_feature_semantics.csv ✅
